# Phase 3 · S3b — F3 dưới **full-ranking** trên đúng dữ liệu *reviews* (coverage F3 = 100%)

**Lý do:** S3 cho thấy F3 không transfer sang chainRec, nhưng đó là vì chain-structure (interactions) và
review-length (reviews) **lệch nguồn** → coverage F3 trên edge chainRec rất thấp. Câu hỏi còn lại:
**F3 +3.6% của Phase 2 là tín hiệu THẬT hay chỉ là artifact của sampled@500?**

Test quyết định: chạy **ALS trên chính reviews** (nơi F3 phủ 100%) nhưng đổi eval sang **full-ranking**
(cùng `rank_eval` nghiêm như S1a/S2/S3). So 3 biến thể confidence + sàn:
- **vanilla** (conf=1) · **+F3 length** `(rating/5)·log1p(token_count)` · **+votes** `(rating/5)·log1p(n_votes)` · **itemPop**

Kết luận khả dĩ:
- F3 > vanilla ở full-ranking ⇒ F3 **thật** (chỉ không tới chainRec do lệch dữ liệu).
- F3 phẳng ⇒ +3.6% là artifact sampled@500.
- votes < vanilla ⇒ tái xác nhận votes gây popularity bias.

**Nền tảng:** Kaggle GPU T4 + Internet On. Dùng đúng universe reviews (string id), split thời gian như Phase 2.


## 0 · Setup

In [ ]:
import os, json, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch
try:
    import implicit
except ImportError:
    os.system("pip install -q implicit"); import implicit
from scipy.sparse import csr_matrix

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| implicit", implicit.__version__)

HF_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN=UserSecretsClient().get_secret("HF_TOKEN")
except Exception: pass

HF_REPO="vngclinh/goodreads-preprocessed"
S3B_LOCAL=Path("/kaggle/working/s3b"); S3B_LOCAL.mkdir(parents=True, exist_ok=True)
EVAL_USERS=5000; BATCH_USERS=64; K_LIST=(10,20)
from huggingface_hub import hf_hub_download, HfApi
def _hf(rel): return hf_hub_download(HF_REPO, rel, repo_type="dataset", token=HF_TOKEN)

## 1 · Load reviews (mọi genre) + build index + split thời gian

Positive = `rating>=4` (giống Phase 2). Train trên split train, test trên split test.
Index user/book xây trên train-positive (chỉ rank được item có factor).

In [ ]:
api=HfApi()
parq=sorted(f for f in api.list_repo_files(HF_REPO,repo_type="dataset")
            if f.startswith("data/") and f.endswith(".parquet"))
cols=["user_id","book_id","rating","review_token_count","n_votes","split"]
parts=[]
for f in parq:
    d=pd.read_parquet(_hf(f), columns=cols)
    parts.append(d); print(f"  {Path(f).stem:42} {len(d):,}")
df=pd.concat(parts, ignore_index=True)
df["user_id"]=df["user_id"].astype(str); df["book_id"]=df["book_id"].astype(str)
df["n_votes"]=pd.to_numeric(df["n_votes"], errors="coerce").fillna(0.0)
df["review_token_count"]=pd.to_numeric(df["review_token_count"], errors="coerce").fillna(0.0)
print("split values:", df["split"].value_counts().to_dict())

tr  = df[(df["split"]=="train") & (df["rating"]>=4)].copy()
te  = df[(df["split"]=="test")  & (df["rating"]>=4)].copy()
all_u=sorted(tr["user_id"].unique()); all_b=sorted(tr["book_id"].unique())
u2i={u:i for i,u in enumerate(all_u)}; b2i={b:i for i,b in enumerate(all_b)}
N_U,N_I=len(all_u),len(all_b)
print(f"train-pos={len(tr):,}  test-pos={len(te):,}  N_U={N_U:,}  N_I={N_I:,}")

tr["ui"]=tr["user_id"].map(u2i); tr["bi"]=tr["book_id"].map(b2i)
te=te[te["user_id"].isin(u2i) & te["book_id"].isin(b2i)].copy()
te["ui"]=te["user_id"].map(u2i); te["bi"]=te["book_id"].map(b2i)
print(f"test-pos rankable (user&book có trong train): {len(te):,}")

## 2 · `rank_eval` full-ranking + scorers (giống S1a/S2/S3)

In [ ]:
@torch.no_grad()
def make_als_scorer(U,V,device="cuda"):
    Ut=torch.as_tensor(U,dtype=torch.float32,device=device); Vt=torch.as_tensor(V,dtype=torch.float32,device=device)
    def fn(u): return Ut[u]@Vt.t()
    return fn
@torch.no_grad()
def make_pop_scorer(pop,device="cuda"):
    p=torch.as_tensor(pop,dtype=torch.float32,device=device)
    def fn(u): return p.unsqueeze(0).expand(u.shape[0],-1)
    return fn

@torch.no_grad()
def rank_eval(score_fn, test_pairs, user_item_map, n_item, pos_stage=None,
              K_list=(10,20), batch_users=64, n_eval_users=None, device="cuda", seed=999):
    pairs=test_pairs if pos_stage is None else test_pairs[test_pairs[:,2]==pos_stage]
    if n_eval_users is not None and len(pairs)>n_eval_users:
        rng=np.random.default_rng(seed); pairs=pairs[rng.choice(len(pairs),size=n_eval_users,replace=False)]
    aucs=[]; hits={k:[] for k in K_list}; ndcg={k:[] for k in K_list}
    for st in range(0,len(pairs),batch_users):
        chunk=pairs[st:st+batch_users]
        u=torch.tensor(chunk[:,0],dtype=torch.long,device=device); pos=torch.tensor(chunk[:,1],dtype=torch.long,device=device)
        scores=score_fn(u); B=u.shape[0]; ar=torch.arange(B,device=device)
        pos_score=scores[ar,pos].clone(); seen_cnt=torch.zeros(B,device=device)
        for b in range(B):
            seen=user_item_map.get(int(u[b]),())
            if seen:
                idx=torch.tensor(list(seen),dtype=torch.long,device=device); scores[b,idx]=float("-inf"); seen_cnt[b]=len(seen)
        scores[ar,pos]=pos_score
        rank=(scores>pos_score.unsqueeze(1)).sum(1).float(); neg=(n_item-seen_cnt).clamp(min=1)
        aucs.append((1.0-rank/neg).cpu().numpy()); rnp=rank.cpu().numpy()
        for k in K_list:
            hit=rnp<k; hits[k].append(hit.astype(float)); ndcg[k].append(np.where(hit,1.0/np.log2(rnp+2),0.0))
        del scores
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    res={"AUC":float(np.concatenate(aucs).mean()),"n_eval":int(len(pairs))}
    for k in K_list:
        res[f"Recall@{k}"]=float(np.concatenate(hits[k]).mean()); res[f"NDCG@{k}"]=float(np.concatenate(ndcg[k]).mean())
    return res

# mask = train-positive books mỗi user; test_pairs = test positives
from collections import defaultdict
uim=defaultdict(set)
for ui,bi in zip(tr["ui"].values, tr["bi"].values): uim[int(ui)].add(int(bi))
uim=dict(uim)
test_pairs=np.stack([te["ui"].values, te["bi"].values, np.zeros(len(te),dtype=np.int64)],axis=1).astype(np.int64)
print("test_pairs:", test_pairs.shape, "| users w/ mask:", len(uim))

## 3 · Train ALS 3 biến thể confidence + eval full-ranking

`vanilla=1` · `f3=(rating/5)·log1p(token_count)` · `votes=(rating/5)·log1p(n_votes)`. Hyperparams khớp Phase 2.

In [ ]:
rows=tr["ui"].to_numpy(); cols=tr["bi"].to_numpy()
assert not (np.isnan(rows).any() or np.isnan(cols).any()), "NaN trong index (user/book chưa có trong train index)"
rows=rows.astype(np.int32); cols=cols.astype(np.int32)
rt=tr["rating"].astype(float).to_numpy()
raw_W={
  "vanilla":      np.ones(len(tr),dtype=np.float64),
  "+F3(length)":  (rt/5.0)*np.log1p(np.nan_to_num(tr["review_token_count"].astype(float).to_numpy())),
  "+votes":       (rt/5.0)*np.log1p(np.nan_to_num(tr["n_votes"].astype(float).to_numpy())),
}

ALPHA=40.0; CAP=50.0
def prep_conf(raw):
    # EQUAL-MASS confidence: C = 1 + ALPHA*(w/mean(w)) -> mọi biến thể cùng confidence-mass TRUNG BÌNH,
    # chỉ khác PHÂN PHỐI => cô lập 'hình dạng trọng số' khỏi 'độ lớn'. cap chặn đuôi -> không NaN.
    w=np.nan_to_num(np.asarray(raw,dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    w=np.clip(w,0.0,None)
    hi=np.quantile(w,0.999); w=np.minimum(w,hi)          # winsorize đuôi (review viral)
    mu=w.mean()
    w=np.ones_like(w) if mu<=0 else w/mu                 # mean=1 (bằng mass giữa các biến thể)
    w=np.minimum(w,CAP)                                  # cap tỉ lệ -> ổn định số học
    return (1.0 + ALPHA*w).astype(np.float32)

def build_m(conf):
    m=csr_matrix((conf,(rows,cols)),shape=(N_U,N_I),dtype=np.float32)
    m.sum_duplicates(); m.eliminate_zeros(); return m

# --- chẩn đoán ma trận (in 1 lần trên vanilla) ---
m0=build_m(prep_conf(raw_W["vanilla"]))
rsum=np.asarray(m0.sum(1)).ravel(); csum=np.asarray(m0.sum(0)).ravel()
print(f"matrix {m0.shape}  nnz={m0.nnz:,}  empty_rows={(rsum==0).sum():,}  empty_cols={(csum==0).sum():,}")
print(f"conf finite={np.isfinite(m0.data).all()}  min/max={m0.data.min():.3g}/{m0.data.max():.3g}")

def fit_als(vals):
    m=build_m(prep_conf(vals))
    for r in (0.1, 1.0, 10.0, 100.0):     # nâng regularization tới khi hết NaN
        a=implicit.als.AlternatingLeastSquares(factors=64,iterations=20,regularization=r,
                                               random_state=SEED,use_gpu=False)
        try:
            t0=time.time(); a.fit(m)
            U=np.asarray(a.user_factors,dtype=np.float32); V=np.asarray(a.item_factors,dtype=np.float32)
            if np.isfinite(U).all() and np.isfinite(V).all():
                print(f"   fit {time.time()-t0:.1f}s (reg={r})"); return U,V
            print(f"   NaN/inf ở reg={r} -> nâng reg")
        except Exception as e:
            print(f"   {type(e).__name__} ở reg={r}: {e} -> nâng reg")
    raise RuntimeError("ALS vẫn NaN dù regularization tới 100 — xem chẩn đoán empty_rows/cols ở trên")

item_pop=np.bincount(cols,minlength=N_I).astype(np.float32)
res={}
for name,vals in raw_W.items():
    print(f"[ALS {name}] training ...")
    U,V=fit_als(vals)
    r=rank_eval(make_als_scorer(U,V,device=DEVICE), test_pairs, uim, N_I, pos_stage=None,
                K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)
    res[f"ALS {name}"]=r
    print(f"   AUC={r['AUC']:.4f}  R@10={r['Recall@10']:.4f}  N@10={r['NDCG@10']:.4f}")
res["itemPop (floor)"]=rank_eval(make_pop_scorer(item_pop,device=DEVICE), test_pairs, uim, N_I,
                                 pos_stage=None, K_list=K_LIST, batch_users=BATCH_USERS, n_eval_users=EVAL_USERS, device=DEVICE)

## 4 · Bảng kết quả + Δ + đối chiếu Phase 2 (sampled@500)

In [ ]:
print(f"{'config':22}{'AUC':>9}{'R@10':>9}{'N@10':>9}{'R@20':>9}")
for name,r in res.items():
    print(f"{name:22}{r['AUC']:>9.4f}{r['Recall@10']:>9.4f}{r['NDCG@10']:>9.4f}{r['Recall@20']:>9.4f}")

base=res["ALS vanilla"]
def d(name,m): return f"{res[name][m]-base[m]:+.4f} ({(res[name][m]/base[m]-1)*100:+.1f}%)"
print("\n=== Δ vs ALS vanilla (full-ranking) ===")
for v in ["ALS +F3(length)","ALS +votes"]:
    print(f"{v:18}  AUC {d(v,'AUC')}  | R@10 {d(v,'Recall@10')}  | N@10 {d(v,'NDCG@10')}")

print("\n=== Tham chiếu Phase 2 (sampled@500, KHÁC protocol — chỉ để đối chiếu xu hướng) ===")
print("  F3 length: R@10=0.6122 N@10=0.5369  |  F2 baseline: R@10=0.5757 N@10=0.5107  (F3 +3.6% R@10)")
json.dump(res, open(S3B_LOCAL/"s3b_reviews_fullrank.json","w"), indent=2)
print("\nSaved s3b_reviews_fullrank.json")

## 5 · Push + kết luận

In [ ]:
to_push=sorted(S3B_LOCAL.glob("*"))
if HF_TOKEN and to_push:
    for p in to_push:
        if p.stat().st_size>200e6: print("  skip:",p.name); continue
        api.upload_file(path_or_fileobj=str(p),path_in_repo=f"s3b/{p.name}",repo_id=HF_REPO,repo_type="dataset",token=HF_TOKEN)
        print("  ✓",p.name)
else:
    print("Không push. Artifact ở /kaggle/working/s3b.")

## 6 · Diễn giải (điền sau khi chạy)

- **F3(length) vs vanilla**: dương ⇒ review-length là tín hiệu engagement THẬT, sống sót dưới full-ranking
  (chỉ không tới được chainRec do chain ⟷ review lệch nguồn dữ liệu). Phẳng/âm ⇒ +3.6% của Phase 2 phần lớn
  là artifact của sampled@500.
- **votes vs vanilla**: âm ⇒ tái xác nhận votes gây popularity bias dưới full-ranking (claim 3).
- So sánh độ tụt giữa sampled@500 (Phase 2) và full-ranking ở đây cho thấy sampled@500 thổi phồng cỡ nào.

**Khung đóng góp đề xuất sau S3+S3b (trung thực):**
1. Review-length > votes làm engagement-weight cho MF (cả sampled@500 lẫn full-ranking nếu S3b dương).
2. votes gây popularity bias (âm) — tái lập.
3. Tín hiệu này **không transfer sang chainRec** vì chain-structure (interactions) và review-length (reviews)
   nằm ở hai nguồn chồng lấn thấp — một phát hiện về *giới hạn transfer*, kèm số coverage từ S3.
4. (Độc lập) ALS thuần > chainRec trên Goodreads full-ranking — củng cố nhận định "Goodreads bất lợi cho chainRec" của paper.
